# Mission 10 — Colab 파이프라인 점검 노트

`src/mission10/` 모듈의 **클래스와 함수를 그대로 불러다 쓰면서** 각 단계를 하나씩 눈으로 확인한다.
로직은 이 노트북에 쓰지 않는다. 여기서 문제를 찾으면 고치는 곳은 항상 `src/mission10/*.py` 쪽이다.

## 전체 흐름

```
원본 문서 (20 Newsgroups)
        |
        v
  load_raw_data          preprocessing.py   -> DataFrame(text, label)
        |
        v
  train_test_split_texts preprocessing.py   -> train / val / test
        |
        v
  clean_text + tokenize  preprocessing.py   -> 문서별 토큰 리스트
        |
        v
  build_vocab            preprocessing.py   -> {단어: ID}
        |
        v
  train_word2vec 등      embeddings.py      -> 단어별 벡터
  build_embedding_matrix embeddings.py      -> (vocab_size, embedding_dim)
        |
        v
  build_dataloader       dataset.py         -> 배치 [B, L]
        |
        v
  RNNTextClassifier      model.py           -> 로짓 [B, 20]
        |
        v
  fit / evaluate         train.py           -> 학습 + 예측
        |
        v
  compute_metrics        metrics.py         -> accuracy, f1 ...
        |
        v
  compare_results        compare.py         -> 임베딩 방식별 비교표
```

## 사용법

위에서부터 순서대로 실행한다. 각 단계 끝에 **`[확인]`** 으로 시작하는 출력이 있는데,
그 값이 예상과 다르면 거기서 멈추고 해당 모듈을 고친 뒤 다시 돌린다.

처음에는 `SAMPLE_N`을 작게 두고 전체 흐름이 끝까지 도는지부터 본다.
흐름이 확인되면 `SAMPLE_N = None`으로 바꿔 전체 데이터로 돌린다.

> **런타임**: 메뉴에서 `런타임 > 런타임 유형 변경 > T4 GPU`를 먼저 선택한다. CPU로도 돌아가지만 학습이 느리다.

---
## 0. 환경 준비

### 0-1. 저장소 가져오기

저장소가 public이라 인증 없이 clone된다.

`BRANCH`는 **어느 브랜치의 모듈 코드를 불러올지** 정한다.

PR #10이 main에 병합되어 지금은 `main`에 모든 모듈이 구현되어 있다.
모듈을 고치는 중이라면 그 브랜치 이름으로 바꿔서 고친 코드를 바로 검증할 수 있다.


In [ ]:
BRANCH = "main"   # 모듈을 고치는 중이면 그 브랜치 이름으로 바꾼다
REPO   = "https://github.com/mingilee97/Mission_10.git"
ROOT   = "/content/Mission_10"

import os, shutil, subprocess

if os.path.exists(ROOT):
    shutil.rmtree(ROOT)          # 매번 깨끗한 상태에서 시작 (수정본이 있으면 먼저 push할 것)

subprocess.run(["git", "clone", "--branch", BRANCH, REPO, ROOT], check=True)

print("[확인] 체크아웃한 브랜치와 커밋:")
subprocess.run(["git", "-C", ROOT, "log", "--oneline", "-1"])
subprocess.run(["git", "-C", ROOT, "branch", "--show-current"])

### 0-2. 패키지 설치

`requirements.txt`는 **그대로 쓰지 않는다.** 그 파일은 `torch==2.13.0+cu130`으로 고정되어 있는데,
이건 RTX 50xx 로컬 GPU용이라 Colab에 이미 깔린 torch를 지우고 다시 깔게 된다.
Colab에는 torch / pandas / numpy / scikit-learn / matplotlib이 이미 있으므로 **없는 것만** 설치한다.

In [ ]:
# Colab에 기본 탑재되지 않은 것만 설치
%pip install -q "gensim>=4.3.3" nltk

import importlib, sys

print("[확인] 설치된 버전")
for name in ["numpy", "scipy", "pandas", "sklearn", "matplotlib", "torch", "gensim", "nltk"]:
    try:
        m = importlib.import_module(name)
        print(f"  OK  {name:12s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"  X   {name:12s} {type(e).__name__}: {e}")

> **여기서 `numpy` 관련 에러가 나면**: gensim 설치 과정에서 numpy 버전이 바뀐 것이다.
> `런타임 > 세션 다시 시작`을 누른 뒤, **0-2 셀부터** (0-1은 건너뛰고) 다시 실행하면 된다.
> 세션을 다시 시작해도 `/content/Mission_10`은 남아 있다.

### 0-3. NLTK 데이터 내려받기

`preprocessing.py`가 두 가지를 쓴다.

- `clean_text` -> `stopwords` (영어 불용어 목록)
- `tokenize` -> `word_tokenize` -> `punkt` 문장 분리 모델

NLTK 3.8.2부터 `word_tokenize`가 `punkt` 대신 **`punkt_tab`** 을 찾는다.
버전에 따라 둘 중 뭐가 필요한지 달라서 둘 다 받아둔다.

> 불용어 개수는 NLTK 버전에 따라 다르다 (예전 자료에는 179개로 나오지만 최근 버전은 198개다).
> 개수 자체보다 **에러 없이 불러와지는지**가 확인 지점이다.

In [ ]:
import nltk

for pkg in ["stopwords", "punkt", "punkt_tab"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"  (건너뜀) {pkg}: {e}")

# 실제로 쓰이는 두 함수가 동작하는지 여기서 바로 확인한다
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

print("[확인] 불용어 개수:", len(stopwords.words("english")))
print("[확인] 토큰화 결과:", word_tokenize("Claude is testing the tokenizer."))

### 0-4. 모듈 import + 실행 환경 고정

`src/`를 `sys.path`에 넣어야 `from mission10 import ...`이 동작한다.

그리고 **난수 시드를 고정한다.** 이게 왜 중요하냐면, 이 미션의 목적이
"word2vec / fasttext / glove 중 뭐가 더 나은가"를 비교하는 것이기 때문이다.
시드를 고정하지 않으면 두 실험의 점수 차이가 **임베딩 방식 때문인지 가중치 초기화 난수 때문인지 구분할 수 없다.**

`config.py`에 `train.seed` 필드가 있지만 현재 `train.py` / `compare.py` 어디에서도 쓰이지 않는다.
일단 노트북에서 고정해두고, 나중에 `train.fit()` 안으로 옮기는 게 맞다. (아래 12번 섹션 참고)

In [ ]:
import sys, random
import numpy as np
import torch

SRC = "/content/Mission_10/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from mission10.config import load_config
from mission10 import preprocessing, embeddings, dataset, model, train, metrics, compare, visualize

CONFIG_DIR = "/content/Mission_10/configs"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int) -> None:
    """파이썬 / numpy / torch 난수를 한꺼번에 고정한다."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


print("[확인] device:", device)
if device.type == "cuda":
    print("[확인] GPU:", torch.cuda.get_device_name(0))
else:
    print("      (CPU다. 런타임 유형을 T4 GPU로 바꾸면 학습이 훨씬 빠르다)")
print("[확인] import된 모듈:", [m.__name__.split(".")[-1] for m in
      (preprocessing, embeddings, dataset, model, train, metrics, compare, visualize)])

---
## 1. 설정 로드 — `config.py`

`load_config(base, exp)`는 `base.yaml`을 읽고 그 위에 `exp/*.yaml`의 값만 덮어쓴다.
코드에 하이퍼파라미터를 하드코딩하지 않기 위한 장치다.

여기서 `SAMPLE_N`도 정한다. 이건 config가 아니라 **노트북에서만 쓰는 값**으로,
전체 18,846개 문서 대신 일부만 써서 파이프라인을 빠르게 한 바퀴 돌려보기 위한 것이다.

In [ ]:
config = load_config(f"{CONFIG_DIR}/base.yaml")

# 처음에는 작게 두고 흐름부터 확인한다. 전체로 돌릴 때는 None.
SAMPLE_N = 2000

print("[확인] 설정값")
for section in ("data", "preprocessing", "embedding", "model", "train"):
    print(f"  [{section}]")
    for k, v in vars(getattr(config, section)).items():
        print(f"    {k:20s} = {v}")
print()
print(f"[확인] SAMPLE_N = {SAMPLE_N}  ({'일부만 사용' if SAMPLE_N else '전체 데이터 사용'})")

> `exp/*.yaml`을 얹으려면 `load_config(f"{CONFIG_DIR}/base.yaml", f"{CONFIG_DIR}/exp/gru.yaml")` 처럼 두 번째 인자를 준다.
> 11번 섹션에서 이 방식으로 임베딩 방식을 바꿔가며 비교한다.

---
## 2. 원본 데이터 — `preprocessing.load_raw_data`

20 Newsgroups는 20개 뉴스그룹에 올라온 글을 모은 데이터셋이다. 즉 **20개 클래스 다중분류** 문제다.

`load_raw_data`는 `remove=("headers", "footers", "quotes")` 옵션으로 부른다.
헤더(보낸 사람, 제목), 푸터(서명), 인용문을 지운다는 뜻인데,
안 지우면 모델이 본문 내용이 아니라 "이 뉴스그룹에만 나오는 이메일 주소" 같은 걸 외워버려서
정확도가 비현실적으로 높게 나오기 때문이다.

첫 실행 때는 scikit-learn이 데이터를 내려받느라 1~2분 걸린다.

In [ ]:
import pandas as pd

df = preprocessing.load_raw_data(config.data)

print("[확인] shape:", df.shape, "  (문서 수, 열 수)")
print("[확인] 열:", list(df.columns))
print("[확인] 클래스 수:", df["label"].nunique())
print()
print("[확인] 클래스별 문서 수 (상위 5개 / 하위 5개)")
counts = df["label"].value_counts().sort_values(ascending=False)
print(counts.head().to_string())
print("  ...")
print(counts.tail().to_string())
print()
print(f"[확인] 가장 많은 클래스 {counts.max()}개 vs 가장 적은 클래스 {counts.min()}개 "
      f"-> 비율 {counts.max()/counts.min():.2f}배")

**[여기서 볼 것]**

- 문서 수가 18,846인가? (20 Newsgroups 전체 크기)
- 클래스 불균형이 심하지 않은가? 최대/최소 비율이 2배 이내면 `accuracy`를 그대로 봐도 큰 왜곡은 없다.
  많이 치우쳐 있다면 `metrics.py`가 쓰는 `macro` 평균이 더 중요해진다.

In [ ]:
# 원본 문서 하나를 눈으로 본다
sample_text = df["text"].iloc[7]
print("길이:", len(sample_text), "자")
print("-" * 60)
print(sample_text[:600])

> `remove=(...)` 때문에 **내용이 통째로 비어버린 문서**가 생긴다. 인용문과 서명밖에 없던 글이 그렇다.
> 이건 3번 섹션에서 숫자로 확인한다.

---
## 3. 분리 먼저 — `preprocessing.train_test_split_texts`

`compare.py`의 순서를 그대로 따른다. **정제/토큰화보다 분리를 먼저** 한다.

왜 순서가 이런가 하면, 뒤에서 만들 vocab과 임베딩을 **train 데이터만으로** 만들어야 하기 때문이다.
test 문서에 있는 단어까지 미리 알고 사전을 만들면, 모델이 시험 문제를 미리 본 셈이 된다.
이걸 **data leakage(정보 누출)** 라고 한다.

```
전체 (100%)
   |
   +-- test_ratio=0.2 -->  test (20%)
   |
   +-- 나머지 train_val (80%)
           |
           +-- val_ratio=0.1 --> val  (80% x 0.1 = 8%)
           |
           +-------------------> train (80% x 0.9 = 72%)
```

In [ ]:
texts  = df["text"].tolist()
labels = df["label"].tolist()

if SAMPLE_N is not None:
    # 앞에서부터 자르면 클래스가 치우치므로 라벨 비율을 유지한 채 표본을 뽑는다
    from sklearn.model_selection import train_test_split as _tts
    texts, _, labels, _ = _tts(
        texts, labels, train_size=SAMPLE_N, random_state=config.data.seed, stratify=labels
    )
    print(f"[확인] 표본 {SAMPLE_N}개만 사용")

(train_texts, val_texts, test_texts,
 train_labels, val_labels, test_labels) = preprocessing.train_test_split_texts(
    texts, labels, config.data.test_ratio, config.data.val_ratio, config.data.seed
)

total = len(texts)
print(f"[확인] 전체 {total}개")
for name, part in [("train", train_texts), ("val", val_texts), ("test", test_texts)]:
    print(f"  {name:6s} {len(part):6d}개  ({len(part)/total*100:5.1f}%)")

# stratify가 실제로 먹었는지 확인: 각 split의 클래스 분포가 비슷해야 한다
print()
print("[확인] 클래스 0의 비율 (세 split이 비슷해야 정상)")
for name, part in [("train", train_labels), ("val", val_labels), ("test", test_labels)]:
    print(f"  {name:6s} {part.count(0)/len(part)*100:5.2f}%")

---
## 4. 정제와 토큰화 — `preprocessing.clean_text`, `preprocessing.tokenize`

두 함수가 하는 일을 문서 하나로 먼저 본다.

- `clean_text`: 소문자화 -> 영문자/공백 외 전부 제거 -> 불용어 제거
- `tokenize`: NLTK `word_tokenize`로 단어 단위로 자름

In [ ]:
raw = train_texts[0]
cleaned = preprocessing.clean_text(raw, config.preprocessing)
tokens  = preprocessing.tokenize(cleaned)

print("[원본]")
print(repr(raw[:300]))
print()
print("[clean_text 이후]")
print(repr(cleaned[:300]))
print()
print("[tokenize 이후] 토큰 수:", len(tokens))
print(tokens[:30])

**[여기서 볼 것]**

`clean_text`가 `re.sub(r"[^a-zA-Z\s]", "", text)`로 **숫자를 전부 지운다.**
`"windows 95"` -> `"windows"`가 된다. 뉴스그룹 글에서 버전 번호나 연도가 단서일 수 있는데
그 정보가 사라진다는 뜻이다. 의도한 것인지 확인이 필요하다.

> 이 영향이 생각보다 크다. 숫자를 **지우는 게 아니라 단어에서 떼어내기만** 하므로
> `"ibm3090"`, `"v2"`, `"x11r5"` 같은 토큰이 전부 `"ibm"`, `"v"`, `"xr"`로 뭉개진다.
> 서로 다른 단어가 같은 토큰이 되어버리는 것이다. 아래 셀에서 그 규모를 직접 센다.

또 하나. `clean_text`는 호출될 때마다 `stopwords.words("english")`로 불용어 목록을 새로 만든다.
문서 18,846개면 그 작업을 18,846번 반복한다. 아래에서 시간을 재본다.

In [ ]:
# 숫자 제거가 실제로 얼마나 단어를 뭉개는지 센다
import re
from collections import defaultdict

collapsed = defaultdict(set)
for t in train_texts[:2000]:
    for w in t.lower().split():
        if any(ch.isdigit() for ch in w):
            after = re.sub(r"[^a-zA-Z]", "", w)
            if after:
                collapsed[after].add(w)

merged = {k: v for k, v in collapsed.items() if len(v) > 1}
print(f"[확인] 숫자를 포함한 원본 단어가 정제 후 같은 토큰으로 합쳐진 경우: {len(merged)}건")
print()
for k, v in sorted(merged.items(), key=lambda kv: -len(kv[1]))[:10]:
    print(f"  {k:12s} <- {sorted(v)[:6]}")

In [ ]:
import time

def to_tokens(raw_texts):
    """compare.py의 to_tokens와 같은 방식: 정제 후 토큰화."""
    return [preprocessing.tokenize(preprocessing.clean_text(t, config.preprocessing))
            for t in raw_texts]

t0 = time.time()
train_tokens = to_tokens(train_texts)
elapsed = time.time() - t0

val_tokens  = to_tokens(val_texts)
test_tokens = to_tokens(test_texts)

print(f"[확인] train {len(train_texts)}개 정제+토큰화에 {elapsed:.1f}초")
print(f"       문서당 {elapsed/len(train_texts)*1000:.1f}ms")
if SAMPLE_N is not None:
    print(f"       -> 전체 18,846개면 대략 {elapsed/len(train_texts)*18846:.0f}초 예상")

In [ ]:
import numpy as np

lengths = np.array([len(t) for t in train_tokens])
empty   = int((lengths == 0).sum())

print("[확인] train 토큰 길이 분포")
print(f"  문서 수     {len(lengths)}")
print(f"  평균        {lengths.mean():.1f}")
print(f"  중앙값      {np.median(lengths):.0f}")
print(f"  95분위수    {np.percentile(lengths, 95):.0f}   <- base.yaml의 max_len={config.preprocessing.max_len} 근거")
print(f"  최대        {lengths.max()}")
print()
print(f"[확인] 토큰이 0개인 문서: {empty}개 ({empty/len(lengths)*100:.2f}%)")
if empty:
    print("       -> 내용이 통째로 빈 입력이 모델에 들어간다. 아래 7번에서 다시 다룬다.")

**[여기서 볼 것]**

`base.yaml`의 `max_len: 280`이 "훈련 데이터 토큰 길이 95분위수 기준"이라고 주석에 적혀 있다.
위에서 나온 95분위수와 실제로 맞는지 확인한다. 표본만 썼다면 값이 다를 수 있다.

**토큰이 0개인 문서**가 나왔다면 짚고 넘어가야 한다.
`remove=("headers","footers","quotes")`로 본문이 사라진 문서인데,
이 문서는 나중에 전부 `<pad>`로만 채워진 입력이 되어 모델에 들어간다.
내용이 없는데 라벨은 붙어 있으니, 모델 입장에서는 학습할 수 없는 잡음이다.
지금은 걸러내지 않고 있다.

---
## 5. 단어 사전 — `preprocessing.build_vocab`

토큰(문자열)을 그대로 신경망에 넣을 수는 없다. 신경망은 숫자만 받는다.
그래서 **각 단어에 정수 ID를 붙인 표**를 만든다. 이게 vocab이다.

```
"computer" -> 42
"windows"  -> 17
```

ID 0번과 1번은 특별한 용도로 미리 예약되어 있다.

- `<pad>` = 0 : 길이가 다른 문서를 같은 길이로 맞출 때 빈자리를 채우는 값
- `<unk>` = 1 : vocab에 없는 단어(unknown)를 대신 가리키는 값

`min_count`는 "몇 번 이상 나온 단어만 사전에 넣을지"를 정한다.
한 번밖에 안 나온 단어까지 다 넣으면 사전이 쓸데없이 커진다.

In [ ]:
vocab = preprocessing.build_vocab(train_tokens, min_count=config.embedding.min_count)

print(f"[확인] vocab 크기: {len(vocab):,}")
print(f"[확인] 특수 토큰: <pad>={vocab['<pad>']}, <unk>={vocab['<unk>']}")
print(f"[확인] 앞쪽 10개: {list(vocab.items())[:10]}")

# min_count를 바꾸면 사전 크기가 어떻게 변하는지
print()
print("[확인] min_count 별 vocab 크기")
for mc in (1, 2, 5, 10):
    print(f"  min_count={mc:2d}  ->  {len(preprocessing.build_vocab(train_tokens, min_count=mc)):,}")

**[여기서 볼 것]**

`compare.py`는 `build_vocab(train_tokens, min_count=config.embedding.min_count)`로 부른다.
그런데 `EmbeddingConfig.min_count`는 원래 **gensim 학습용** 파라미터다
(= "이 횟수보다 적게 나온 단어는 임베딩을 학습하지 않는다").
그걸 vocab 구축에도 그대로 재사용하고 있다.

두 값이 같으면 편한 면은 있다. vocab에 있는 단어는 임베딩도 학습되어 있으니 커버리지가 100%가 된다.
다만 **의도한 설계인지, 그냥 이름이 같아서 가져다 쓴 것인지**는 확인이 필요하다.
따로 두고 싶으면 `PreprocessingConfig`에 `vocab_min_count`를 새로 만드는 게 맞다.

지금 `base.yaml`은 `min_count: 1`이라 **모든 단어가 사전에 들어간다.** 위 표에서 그 차이가 보인다.

---
## 6. 임베딩 — `embeddings.py`

### 6-1. 임베딩이 뭔가

vocab은 단어를 정수로 바꿔줬을 뿐이다. 그런데 정수 ID에는 의미가 없다.
`"computer"=42`, `"windows"=17`이라고 해서 42와 17 사이에 어떤 관계가 있는 건 아니다.

**임베딩(embedding)은 각 단어를 "의미가 담긴 숫자 벡터"로 바꾸는 것**이다.
비슷한 뜻의 단어끼리는 벡터도 비슷해지도록 학습한다.

```
"computer" -> [0.21, -0.53, 0.11, ... ]   <- 100개의 숫자
"pc"       -> [0.19, -0.49, 0.14, ... ]   <- computer와 비슷한 방향
"banana"   -> [-0.71, 0.33, -0.62, ... ]  <- 전혀 다른 방향
```

이 미션에서 비교하는 세 가지 방식은 이렇다.

| 방식 | 어떻게 만드나 | 특징 |
|---|---|---|
| **Word2Vec** | 우리 데이터로 직접 학습 | 이 데이터에 특화됨. 모르는 단어는 처리 못 함 |
| **FastText** | 우리 데이터로 직접 학습 | 단어를 글자 조각으로 쪼개서 학습 -> 처음 보는 단어도 벡터를 만들어냄 |
| **GloVe** | 이미 학습된 파일을 내려받아 사용 | 훨씬 큰 데이터로 학습됨. 우리 데이터에만 있는 단어는 없음 |

### 6-2. Word2Vec 학습

`train_word2vec`은 **train 토큰만** 넣어서 학습한다 (val/test 정보 누출 방지).

In [ ]:
set_seed(config.train.seed)

t0 = time.time()
w2v = embeddings.train_word2vec(
    train_tokens,
    config.embedding.embedding_dim,
    config.embedding.window,
    config.embedding.min_count,
)
print(f"[확인] 학습 시간 {time.time()-t0:.1f}초")
print(f"[확인] 학습된 단어 수: {len(w2v.wv):,}")
print(f"[확인] 벡터 차원: {w2v.wv.vector_size}  (base.yaml의 embedding_dim={config.embedding.embedding_dim})")

학습이 제대로 됐는지 보는 가장 직관적인 방법은 **비슷한 단어를 뽑아보는 것**이다.
의미가 통하는 단어들이 나오면 학습이 된 것이고, 뒤죽박죽이면 데이터가 부족하거나 학습이 덜 된 것이다.

In [ ]:
for word in ["computer", "god", "car", "game"]:
    if word in w2v.wv:
        similar = w2v.wv.most_similar(word, topn=5)
        print(f"{word:10s} -> " + ", ".join(f"{w}({s:.2f})" for w, s in similar))
    else:
        print(f"{word:10s} -> vocab에 없음")

**[여기서 볼 것]**

`SAMPLE_N`이 작으면 결과가 엉망일 수 있다. Word2Vec은 데이터가 많아야 의미 있는 벡터를 만든다.
표본 2,000개로는 부족한 게 정상이다. 전체 데이터로 돌렸을 때 개선되는지가 진짜 확인 지점이다.

참고로 `train_word2vec`은 `epochs`를 넘기지 않아 gensim 기본값 5가 쓰인다.
`sg`(skip-gram 사용 여부)도 기본값 0(CBOW)이다. 이 값들을 config로 뺄지는 선택 사항이다.

### 6-3. 임베딩 행렬 만들기 — `build_embedding_matrix`

지금 두 개의 서로 다른 표가 있다.

- `vocab`: 단어 -> 정수 ID (모델이 쓰는 번호)
- `w2v.wv`: 단어 -> 벡터 (의미가 담긴 숫자들)

모델의 임베딩 레이어는 **"ID를 받아서 벡터를 돌려주는 표"** 다.
즉 `[ID번째 줄] = 그 단어의 벡터`인 행렬이 필요하다. 그걸 만드는 게 `build_embedding_matrix`다.

```
행렬 shape = (vocab_size, embedding_dim)

  ID 0 (<pad>)     [0, 0, 0, ... , 0]        <- 항상 0벡터
  ID 1 (<unk>)     [랜덤, 랜덤, ...]
  ID 2 ("computer")[0.21, -0.53, ...]        <- w2v에서 가져옴
  ID 3 ("windows") [0.44, 0.12, ...]
  ...
```

w2v에 없는 단어는 랜덤 초기화된 채로 남는다. 그 비율(**커버리지**)이 중요하다.

In [ ]:
emb_matrix = embeddings.build_embedding_matrix(vocab, config.embedding.embedding_dim, w2v.wv)

print(f"[확인] shape: {emb_matrix.shape}  (vocab_size={len(vocab)}, embedding_dim={config.embedding.embedding_dim})")
print(f"[확인] dtype: {emb_matrix.dtype}   (float32여야 torch로 넘길 때 변환 비용이 없다)")
print(f"[확인] <pad> 행이 0벡터인가: {bool((emb_matrix[vocab['<pad>']] == 0).all())}")
print()

covered = sum(1 for w in vocab if w in w2v.wv)
print(f"[확인] 커버리지: {covered:,} / {len(vocab):,} = {covered/len(vocab)*100:.1f}%")
print("       (나머지는 랜덤값으로 남는다. min_count가 1이면 거의 100%여야 정상)")

**[여기서 볼 것]**

커버리지가 낮으면 임베딩을 쓰는 의미가 줄어든다. 랜덤 벡터가 많다는 뜻이기 때문이다.
`config.embedding.min_count`가 vocab과 gensim 양쪽에 같이 쓰이므로 지금은 거의 100%가 나와야 한다.
GloVe로 바꾸면 이 값이 크게 떨어질 텐데, 그때 이 셀을 다시 보면 된다.

### 6-4. FastText와 비교해보기

FastText의 핵심은 **단어를 글자 조각(subword)으로 쪼개서 학습한다**는 점이다.
`"computer"`를 `"com"`, `"omp"`, `"mpu"` ... 같은 조각들로 나눠서 각 조각의 벡터를 배운다.

덕분에 **학습 때 한 번도 본 적 없는 단어도 벡터를 만들어낼 수 있다.**
조각들을 조합하면 되기 때문이다. Word2Vec은 이게 불가능하다.

이 차이가 실제로 나타나는지 확인해본다.

In [ ]:
set_seed(config.train.seed)
ft = embeddings.train_fasttext(
    train_tokens,
    config.embedding.embedding_dim,
    config.embedding.window,
    config.embedding.min_count,
)

fake_word = "supercalifragilisticexpialidocious"   # 데이터에 있을 리 없는 단어

print(f"[확인] '{fake_word}' 조회")
print(f"  word2vec 에 있나? {fake_word in w2v.wv}")
print(f"  fasttext 에 있나? {fake_word in ft.wv}   <- subword 덕분에 True가 나와야 정상")

if fake_word in ft.wv:
    print(f"  fasttext가 만들어낸 벡터 앞 5개: {ft.wv[fake_word][:5]}")

**[여기서 볼 것]**

`build_embedding_matrix`는 `if word in keyed_vectors:` 로 판단한다.
FastText의 경우 이 `in` 검사가 subword까지 고려해서 True를 돌려주므로 OOV 단어도 벡터를 받는다.
위 출력에서 그게 실제로 그런지 확인된다.

다만 지금 파이프라인에서는 **vocab을 train 데이터로 만들고 임베딩도 train 데이터로 학습**하므로
vocab 안에 OOV 단어가 거의 없다. 즉 **FastText의 최대 장점이 발휘될 상황이 아니다.**
`min_count`를 올려서 희귀 단어를 vocab에서 빼면 그때 차이가 드러날 수 있다.
성능 비교 결과를 해석할 때 이 점을 감안해야 한다.

---
## 7. 배치 만들기 — `dataset.py`

### 7-1. 왜 padding이 필요한가

문서마다 길이가 다르다. 어떤 글은 30단어, 어떤 글은 900단어다.
그런데 GPU는 **직사각형 행렬**을 한 번에 계산할 때 가장 빠르다.
그래서 한 배치 안의 문서들을 **같은 길이로 맞춘다.** 짧은 문서 뒤에 `<pad>`(=0)를 채우는 것이다.

```
원래:
  문서A: [5, 12, 8]
  문서B: [3, 9, 21, 7, 15]

padding 후 (길이 5로 통일):
  문서A: [5, 12, 8, 0, 0]
  문서B: [3, 9, 21, 7, 15]
                 ^^^^^^ <pad>=0
```

`collate_fn`이 이 일을 한다. `max_len`보다 긴 문서는 잘라낸다.

In [ ]:
def to_sequences(token_lists):
    """compare.py와 같은 방식: 토큰 -> vocab ID. 없는 단어는 <unk>."""
    unk_id = vocab["<unk>"]
    return [[vocab.get(tok, unk_id) for tok in tokens] for tokens in token_lists]

train_seqs = to_sequences(train_tokens)
val_seqs   = to_sequences(val_tokens)
test_seqs  = to_sequences(test_tokens)

print("[확인] 토큰 -> ID 변환")
print("  토큰:", train_tokens[0][:10])
print("  ID  :", train_seqs[0][:10])

unk_id = vocab["<unk>"]
unk_count = sum(s.count(unk_id) for s in test_seqs)
total_tok = sum(len(s) for s in test_seqs)
print()
print(f"[확인] test 데이터의 <unk> 비율: {unk_count:,} / {total_tok:,} = {unk_count/total_tok*100:.2f}%")
print("       (vocab은 train으로만 만들었으므로 test에는 모르는 단어가 나오는 게 정상이다)")

In [ ]:
train_loader = dataset.build_dataloader(
    train_seqs, train_labels, config.train.batch_size,
    shuffle=True, max_len=config.preprocessing.max_len,
)
val_loader = dataset.build_dataloader(
    val_seqs, val_labels, config.train.batch_size,
    shuffle=False, max_len=config.preprocessing.max_len,
)
test_loader = dataset.build_dataloader(
    test_seqs, test_labels, config.train.batch_size,
    shuffle=False, max_len=config.preprocessing.max_len,
)

# 배치 하나를 꺼내서 shape을 직접 확인한다
batch_x, batch_y = next(iter(train_loader))

print("[확인] 배치 하나")
print(f"  입력 shape : {tuple(batch_x.shape)}   (B=배치크기, L=시퀀스길이)")
print(f"  라벨 shape : {tuple(batch_y.shape)}")
print(f"  입력 dtype : {batch_x.dtype}   (nn.Embedding은 정수 인덱스를 받으므로 int64)")
print(f"  라벨 범위  : {batch_y.min().item()} ~ {batch_y.max().item()}   (0~19 이어야 한다)")
print()
print(f"  기대값: B={config.train.batch_size}, L={config.preprocessing.max_len}")

### 7-2. padding이 실제로 얼마나 차지하나

여기가 이 파이프라인에서 **가장 중요한 점검 지점**이다.

`collate_fn`은 배치를 항상 `max_len`(=280)까지 채운다.
그런데 4번 섹션에서 본 것처럼 **문서 중앙값 길이는 그보다 훨씬 짧다.**
즉 대부분의 자리가 의미 없는 `<pad>`로 채워진다.

In [ ]:
real = (batch_x != vocab["<pad>"]).sum().item()
allc = batch_x.numel()

print(f"[확인] 배치 안에서")
print(f"  전체 칸    {allc:,}")
print(f"  실제 단어  {real:,}  ({real/allc*100:.1f}%)")
print(f"  padding    {allc-real:,}  ({(allc-real)/allc*100:.1f}%)")
print()

per_doc_real = (batch_x != vocab["<pad>"]).sum(dim=1)
print(f"[확인] 문서별 실제 길이: 최소 {per_doc_real.min().item()}, "
      f"중앙값 {per_doc_real.median().item()}, 최대 {per_doc_real.max().item()}")
print(f"       (전부 {config.preprocessing.max_len} 길이로 패딩되어 모델에 들어간다)")

**[여기서 볼 것 — 중요]**

padding 비율이 높다는 건 두 가지 문제를 뜻한다.

**첫째, 계산 낭비.** LSTM이 아무 의미 없는 `<pad>` 자리를 전부 계산한다.

**둘째, 이게 진짜 문제인데 — 모델이 마지막 hidden state를 쓴다.**

`model.py`의 `forward`를 보면 `h_n[-1]`, 즉 **시퀀스를 끝까지 다 처리한 뒤의 상태**를 분류에 쓴다.
그런데 실제 단어가 30개인 문서라면, LSTM은 30번째 이후로 250번의 `<pad>`를 더 처리한 상태를 넘긴다.
문서의 진짜 내용이 250스텝의 padding에 희석되는 것이다.

```
실제 내용                        padding
[w1 w2 ... w30][pad pad pad ... pad]  ->  h_n[-1] 을 분류에 사용
              ^                      ^
              여기서 뽑아야 함        실제로 뽑는 위치
```

**표준 해법은 `nn.utils.rnn.pack_padded_sequence`** 다.
각 문서의 진짜 길이를 함께 넘겨서 LSTM이 padding을 건너뛰게 하는 방법이다.
그러면 `h_n`이 각 문서의 마지막 진짜 단어 시점의 상태가 된다.

이걸 적용하려면 `collate_fn`이 길이도 함께 돌려주고, `forward`가 그 길이를 받아야 한다.
**9번 섹션에서 학습을 돌려본 뒤, 정확도가 기대보다 낮으면 여기부터 손보는 게 맞다.**

---
## 8. 모델 — `model.RNNTextClassifier`

### 8-1. 구조

```
입력 ID          [B, 280]
    |
    v
nn.Embedding     [B, 280, 100]     <- 각 ID를 100차원 벡터로
    |
    v
LSTM (2층, 양방향)
    |
    +--> h_n     [4, B, 128]       <- 2층 x 2방향 = 4
    |
    v
마지막 층의 정/역방향 이어붙임
                 [B, 256]          <- 128 x 2
    |
    v
nn.Linear        [B, 20]           <- 20개 클래스 점수(로짓)
```

`bidirectional=True`는 문장을 **앞에서 뒤로, 뒤에서 앞으로 두 번** 읽는다는 뜻이다.
그래서 hidden이 두 개 나오고, 이어붙이면 차원이 2배가 된다.

In [ ]:
set_seed(config.train.seed)

num_classes = df["label"].nunique()

clf = model.RNNTextClassifier(
    emb_matrix,
    config.model,
    num_classes,
    pad_id=vocab["<pad>"],
    freeze_embedding=config.embedding.freeze,
).to(device)

print(clf)
print()
total_params     = sum(p.numel() for p in clf.parameters())
trainable_params = sum(p.numel() for p in clf.parameters() if p.requires_grad)
emb_params       = clf.embedding.weight.numel()

print(f"[확인] 전체 파라미터   {total_params:,}")
print(f"[확인] 학습 대상       {trainable_params:,}")
print(f"[확인] 임베딩이 차지   {emb_params:,}  ({emb_params/total_params*100:.1f}%)")
print(f"       freeze={config.embedding.freeze} -> 임베딩 학습 여부: {clf.embedding.weight.requires_grad}")

In [ ]:
# forward를 한 번 통과시켜 shape이 예상대로인지 본다
clf.eval()
with torch.no_grad():
    logits = clf(batch_x.to(device))

print(f"[확인] 입력  {tuple(batch_x.shape)}")
print(f"[확인] 출력  {tuple(logits.shape)}   기대값: ({config.train.batch_size}, {num_classes})")
print()
print(f"[확인] 첫 문서의 로짓 앞 5개: {logits[0][:5].cpu().numpy().round(3)}")
print("       (아직 학습 전이라 값이 고르게 흩어져 있는 게 정상)")

probs = torch.softmax(logits[0], dim=-1)
print(f"[확인] softmax 후 합: {probs.sum().item():.4f}  (1이어야 정상)")
print(f"[확인] 최대 확률: {probs.max().item():.4f}  (학습 전이면 1/20=0.05 근처)")

**[여기서 볼 것]**

- 출력이 `(B, 20)`인가? `nn.CrossEntropyLoss`는 **softmax를 적용하지 않은 로짓**을 받는다.
  `forward`가 softmax를 붙이지 않고 `Linear` 결과를 그대로 돌려주는 게 맞다. 현재 코드는 맞게 되어 있다.
- 학습 전 최대 확률이 0.05(=1/20) 근처인가? 특정 클래스로 심하게 쏠려 있으면 초기화가 이상한 것이다.
- `dropout=0.5`가 `num_layers=2`이므로 층 사이에 적용된다. `num_layers=1`로 바꾸면
  PyTorch가 dropout을 무시하고 경고를 낸다. `model.py`에 그 처리가 이미 들어가 있다.

---
## 9. 학습 — `train.fit`

`fit`이 하는 일:

```
epoch 반복
  |
  +-- train_one_epoch:  순전파 -> loss 계산 -> 역전파 -> 가중치 갱신
  |
  +-- evaluate:         val 데이터로 loss만 측정 (가중치 갱신 없음)
  |
  +-- history에 기록
```

`optimizer`는 Adam, `criterion`은 `CrossEntropyLoss`가 `fit` 안에서 만들어진다.
학습률은 `config.train.lr`을 쓴다.

처음에는 **epoch를 적게** 두고 loss가 내려가는지부터 본다.

In [ ]:
set_seed(config.train.seed)

# 흐름 확인용으로 epoch를 줄인다. 전체 학습은 아래 셀에서.
import copy
quick_cfg = copy.deepcopy(config.train)
quick_cfg.epochs = 3

t0 = time.time()
history = train.fit(clf, train_loader, val_loader, quick_cfg, device)
print(f"\n[확인] {quick_cfg.epochs} epoch 학습에 {time.time()-t0:.1f}초")

**[여기서 볼 것]**

- `train_loss`가 epoch마다 **내려가는가?** 안 내려가면 학습률(`lr=0.005`)이 너무 크거나 작은 것이다.
- 학습 직후 loss가 `ln(20) = 3.00` 근처에서 시작하는가? 20개 클래스를 무작위로 찍을 때의 loss가 그 값이다.
  거기서 안 내려가면 모델이 아무것도 못 배우고 있다는 뜻이다.
- `val_loss`가 `train_loss`보다 한참 높고 **올라가기 시작하면** 과적합(overfitting)이다.
  그 시점의 epoch가 실제로 쓸 만한 epoch 수다.

`fit`은 현재 **가장 좋은 모델을 저장하지 않는다.** 마지막 epoch의 상태를 그대로 쓴다.
과적합이 보이면 `val_loss`가 가장 낮은 시점의 가중치를 남기는 로직(early stopping)을 넣는 게 좋다.

### 9-2. padding이 학습을 망치는지 A/B로 확인

7-2에서 지적한 문제(`h_n[-1]`이 padding까지 통과한 상태)가 **실제로 성능에 영향을 주는지**
여기서 직접 재본다. 추측으로 코드를 고치지 않기 위한 단계다.

방법은 간단하다. `forward`만 `pack_padded_sequence`로 바꾼 버전을 만들어
**같은 시드, 같은 데이터로** 현재 코드와 나란히 돌린다.

`pack_padded_sequence`가 하는 일:

```
현재:  [w1 w2 w3 pad pad ... pad]  -> LSTM이 280스텝 전부 처리 -> h_n
                                                    ^ padding까지 먹은 상태

pack:  [w1 w2 w3]                  -> LSTM이 3스텝만 처리 -> h_n
                                        ^ 진짜 마지막 단어 시점의 상태
```

> **주의**: 아래 `Packed` 클래스는 **비교용 임시 코드**다. 차이가 확인되면
> 고칠 곳은 `src/mission10/model.py`의 `forward`이고, 이 셀은 지워도 된다.
> `collate_fn`이 길이를 함께 돌려주도록 바꾸는 게 더 깔끔하지만,
> 여기서는 `x != pad_id`로 길이를 역산해 원본 코드를 건드리지 않았다.

In [ ]:
import torch.nn as nn
import copy


class Packed(model.RNNTextClassifier):
    """비교용. forward만 pack_padded_sequence로 교체한 서브클래스."""

    def forward(self, x):
        # padding이 아닌 칸의 개수 = 문서의 진짜 길이. 0이면 pack이 에러를 내므로 최소 1로 둔다
        lengths = (x != 0).sum(dim=1).clamp(min=1).cpu()
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False
        )
        if self.rnn_type == "lstm":
            _, (h_n, _) = self.rnn(packed)
        else:
            _, h_n = self.rnn(packed)
        last = torch.cat([h_n[-2], h_n[-1]], dim=-1) if self.bidirectional else h_n[-1]
        return self.classifier(last)


def run_variant(cls, lr, epochs=5):
    """같은 시드에서 출발해 한 조합을 학습하고 test 지표를 돌려준다."""
    set_seed(config.train.seed)
    m = cls(emb_matrix, config.model, num_classes,
            pad_id=vocab["<pad>"], freeze_embedding=config.embedding.freeze).to(device)
    tc = copy.deepcopy(config.train)
    tc.lr, tc.epochs = lr, epochs
    h = train.fit(m, train_loader, val_loader, tc, device)
    _, yt, yp = train.evaluate(m, test_loader, torch.nn.CrossEntropyLoss(), device)
    return h, metrics.compute_metrics(yt, yp)


rows = []
for name, cls in [("현재 코드", model.RNNTextClassifier), ("pack 적용", Packed)]:
    for lr in (config.train.lr, 0.001):
        print(f"\n--- {name} / lr={lr} ---")
        h, r = run_variant(cls, lr)
        rows.append((name, lr, h["train_loss"][0], h["train_loss"][-1], r["accuracy"], r["f1"]))

print("\n" + "=" * 68)
print(f"{'구성':16s} {'lr':>7s} {'loss 처음':>10s} {'loss 끝':>9s} {'test acc':>10s} {'f1':>8s}")
print("-" * 68)
for n, lr, l0, l1, a, f in rows:
    print(f"{n:16s} {lr:>7g} {l0:10.3f} {l1:9.3f} {a:10.4f} {f:8.4f}")
print("-" * 68)
print(f"{'무작위 기준':16s} {'':>7s} {np.log(num_classes):10.3f} {'':>9s} {1/num_classes:10.4f}")

**[결과를 어떻게 읽나]**

- **`pack 적용` 쪽 accuracy가 뚜렷하게 높다** -> 7-2의 문제가 실제로 있다.
  `model.py`의 `forward`를 고칠 이유가 생긴 것이다.
- **차이가 없거나 오차 수준이다** -> 이 데이터에서는 문제가 아니다. 고치지 않아도 된다.
- **양쪽 다 무작위 기준(0.05) 근처다** -> padding이 아니라 다른 게 문제다.
  `lr` 두 값의 차이를 먼저 본다. 그래도 안 되면 `epochs`를 늘려본다.

> 참고로 합성 데이터로 이 실험을 먼저 해봤는데, 과제가 너무 쉬워서
> 네 조합이 전부 accuracy 1.0에 도달해 **구분이 되지 않았다.**
> 실제 20 Newsgroups 데이터라야 판별이 된다. 즉 위 표가 이 문제의 첫 실측이다.

> 시간이 오래 걸리면 `run_variant(..., epochs=3)`으로 줄이거나
> `SAMPLE_N`을 작게 둔 채로 먼저 경향만 본다.

---
## 10. 평가 — `train.evaluate` + `metrics.compute_metrics`

`evaluate`는 `(평균 loss, 실제 라벨, 예측 라벨)`을 돌려준다.
그 라벨들을 `compute_metrics`에 넣어 지표를 계산한다.

**지표를 왜 4개나 보나?**

- `accuracy` (정확도): 전체 중 맞힌 비율. 가장 직관적이지만 클래스가 치우치면 속기 쉽다.
- `precision` (정밀도): "A라고 예측한 것들 중 진짜 A인 비율"
- `recall` (재현율): "진짜 A인 것들 중 A라고 맞힌 비율"
- `f1`: precision과 recall의 조화평균. 둘 다 괜찮아야 높아진다.

`compute_metrics`는 `average="macro"`를 쓴다.
**클래스 20개 각각의 지표를 구한 뒤 단순 평균**한다는 뜻이다.
문서가 적은 클래스도 많은 클래스와 같은 비중으로 반영된다.

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
test_loss, y_true, y_pred = train.evaluate(clf, test_loader, criterion, device)

result = metrics.compute_metrics(y_true, y_pred)

print(f"[확인] test loss: {test_loss:.4f}")
print("[확인] 지표")
for k, v in result.items():
    print(f"  {k:10s} {v:.4f}")

print()
print(f"[기준] 무작위로 찍으면 accuracy ≈ {1/num_classes:.4f}")
print(f"[기준] 항상 가장 많은 클래스만 답하면 accuracy ≈ {max(y_true.count(c) for c in set(y_true))/len(y_true):.4f}")

**[여기서 볼 것]**

위 두 기준선(baseline)을 못 넘으면 모델이 아무 의미 있는 걸 못 배운 것이다.
`SAMPLE_N=2000`, `epochs=3`이면 낮게 나오는 게 당연하다. 흐름 확인이 목적이므로 여기서는 괜찮다.

`accuracy`와 `f1`이 크게 차이 나면 **특정 클래스만 잘 맞히고 나머지는 못 맞히는** 상태다.
어느 클래스가 문제인지 아래에서 확인한다.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, zero_division=0, digits=3))

---
## 11. 임베딩 방식 비교 — `compare.py`

여기가 이 미션의 **최종 목표**다. Word2Vec / FastText / GloVe 중 뭐가 나은지 비교한다.

`compare.run_experiment(config)`는 2~10번에서 한 일을 **함수 하나로 전부** 수행한다.
`compare.compare_results([...])`가 그 결과들을 표로 묶는다.

**주의**: `run_experiment`는 내부에서 `load_raw_data`부터 다시 부르므로 `SAMPLE_N`이 적용되지 않는다.
아래 셀은 **전체 18,846개 문서로 학습**한다. GPU에서도 시간이 꽤 걸린다.

In [ ]:
import copy

def make_config(method: str, model_type: str = "lstm", epochs: int | None = None):
    """base 설정을 복사해서 임베딩 방식과 모델 타입만 바꾼다."""
    cfg = copy.deepcopy(config)
    cfg.embedding.method = method
    cfg.model.type = model_type
    if epochs is not None:
        cfg.train.epochs = epochs
    return cfg


# 먼저 짧게 돌려서 끝까지 도는지 확인한다. 확인되면 epochs를 늘린다.
EPOCHS = 3

results = []
for method in ["word2vec", "fasttext"]:
    print(f"\n{'='*50}\n{method}\n{'='*50}")
    set_seed(config.train.seed)          # 매 실험마다 같은 시드에서 출발해야 공정한 비교가 된다
    t0 = time.time()
    results.append(compare.run_experiment(make_config(method, epochs=EPOCHS)))
    print(f"-> {time.time()-t0:.1f}초, metrics: {results[-1]['metrics']}")

In [ ]:
comparison_df = compare.compare_results(results)
comparison_df

**[여기서 볼 것 — 비교 결과를 해석할 때 주의할 점]**

**1. 시드가 고정되어 있는가.**
위 셀에서 `set_seed`를 매 실험 앞에 넣었다. 이게 없으면 두 실험의 점수 차이가
임베딩 때문인지 초기화 난수 때문인지 알 수 없다.
다만 `run_experiment` 내부에서도 난수가 쓰이므로, 완전히 엄밀하게 하려면
`train.fit()` 안에서 `config.train.seed`를 쓰도록 고치는 게 맞다. (12번 참고)

**2. 차이가 의미 있는 크기인가.**
accuracy 차이가 1~2%p 정도면 **우연일 수 있다.** 시드를 3개쯤 바꿔가며 돌려서
평균과 편차를 봐야 "정말 더 낫다"고 말할 수 있다.

**3. FastText가 불리한 조건이다.**
6-4에서 봤듯이 지금 구조에서는 OOV 단어가 거의 없어서 FastText의 장점이 발휘되지 않는다.
"FastText가 더 나쁘다"가 아니라 "이 설정에서는 차이가 안 난다"가 정확한 결론이다.

### GloVe 추가하기

GloVe는 학습이 아니라 **이미 만들어진 벡터 파일을 내려받아 쓴다.**
파일이 없으면 아래 셀을 먼저 실행한다 (약 822MB 압축 파일).

In [ ]:
import os, subprocess, zipfile

GLOVE_DIR  = "/content/Mission_10/data/raw"
GLOVE_PATH = f"{GLOVE_DIR}/glove.6B.100d.txt"
GLOVE_URL  = "https://nlp.stanford.edu/data/glove.6B.zip"   # 스탠퍼드 공식 배포처

if not os.path.exists(GLOVE_PATH):
    os.makedirs(GLOVE_DIR, exist_ok=True)
    zip_path = "/content/glove.6B.zip"

    if not os.path.exists(zip_path):
        print("GloVe 내려받는 중... (822MB, 몇 분 걸린다)")
        # 서버가 느리거나 끊기는 경우가 있어 재시도 옵션을 준다
        subprocess.run(
            ["wget", "--tries=3", "--continue", "-O", zip_path, GLOVE_URL],
            check=True,
        )

    # 4개 차원(50/100/200/300d) 중 우리가 쓰는 100d만 푼다
    print("압축 푸는 중...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extract("glove.6B.100d.txt", GLOVE_DIR)
    print("완료")

print(f"[확인] 파일 존재: {os.path.exists(GLOVE_PATH)}")
if os.path.exists(GLOVE_PATH):
    print(f"[확인] 크기: {os.path.getsize(GLOVE_PATH)/1e6:.0f} MB   (100d 기준 약 347MB)")

In [ ]:
# GloVe 커버리지를 먼저 본다. 여기가 낮으면 성능도 낮게 나온다.
t0 = time.time()
glove = embeddings.load_glove(GLOVE_PATH, config.embedding.embedding_dim)
print(f"[확인] 로드 {time.time()-t0:.1f}초, 단어 {len(glove):,}개")

sample_vec = next(iter(glove.values()))
print(f"[확인] 벡터 차원: {sample_vec.shape[0]}  (embedding_dim={config.embedding.embedding_dim}과 같아야 한다)")

covered = sum(1 for w in vocab if w in glove)
print(f"[확인] 우리 vocab 커버리지: {covered:,}/{len(vocab):,} = {covered/len(vocab)*100:.1f}%")
print("       word2vec은 거의 100%였다. GloVe는 우리 데이터에만 있는 단어를 모르므로 더 낮다.")

In [ ]:
cfg_glove = make_config("glove", epochs=EPOCHS)
cfg_glove.embedding.glove_path = GLOVE_PATH

set_seed(config.train.seed)
results.append(compare.run_experiment(cfg_glove))

comparison_df = compare.compare_results(results)
comparison_df

---
## 12. 시각화 — `visualize.py`

In [ ]:
fig = visualize.plot_training_curves(history, title="LSTM + word2vec")
fig

In [ ]:
fig = visualize.plot_embedding_comparison(comparison_df, metric="accuracy")
fig

In [ ]:
# f1으로도 본다. accuracy와 순위가 다르면 클래스별 편차가 크다는 뜻이다.
fig = visualize.plot_embedding_comparison(comparison_df, metric="f1")
fig

---
## 13. 점검 결과 정리

노트북을 돌리면서 확인된 것들을 여기 적어둔다.
고칠 곳은 전부 `src/mission10/*.py`이고, 이 노트북에는 로직을 넣지 않는다.

### 점검 대상

아래는 **코드를 읽고 판단한 추정**이다. 확정된 사실이 아니므로
각 항목의 '확인' 열에 적힌 셀을 돌려서 실제로 문제인지 먼저 판별한다.

| # | 위치 | 내용 | 어디서 확인 |
|---|---|---|---|
| 1 | `model.py` `forward` | `h_n[-1]`이 padding까지 처리한 뒤의 상태다. `pack_padded_sequence` 필요 | 7-2, **9-2 A/B** |
| 2 | `train.py` `fit` | `config.train.seed`를 쓰지 않는다. 임베딩 비교의 공정성 문제 | 0-4, 11 |
| 3 | `embeddings.py` | 네 함수의 docstring이 삭제됐다. `CLAUDE.md` 규약 위반 | — |
| 4 | `preprocessing.py` `clean_text` | 호출마다 `stopwords.words()`를 새로 만든다 | 4 |
| 5 | `preprocessing.py` `clean_text` | 숫자를 전부 지운다. 의도 확인 필요 | 4 |
| 6 | `compare.py` | `embedding.min_count`를 vocab 구축에 재사용한다 | 5 |
| 7 | 파이프라인 전반 | 토큰이 0개인 문서를 거르지 않는다 | 4 |
| 8 | `train.py` `fit` | 최선 모델을 저장하지 않는다 (early stopping 없음) | 9 |
| 9 | `embeddings.py` | `epochs`, `sg` 등 gensim 파라미터가 config에 없다 | 6-2 |
| 10 | `preprocessing.py` | `min_token_len` 설정이 어디에서도 쓰이지 않는다 (README '알려진 한계'에서 지적) | 4 |

### 우선순위 제안

**2번(시드)이 먼저다.** 이건 성능 문제가 아니라 **비교 실험의 결론 자체를 믿을 수 없게 만드는**
문제이기 때문이다. 시드가 안 잡힌 상태의 비교표는 몇 번을 돌려도 근거가 되지 못한다.

**1번은 9-2의 A/B 결과를 보고 판단한다.** 차이가 없으면 고칠 이유가 없다.

3번은 규약 문제라 언제 고쳐도 되지만 잊기 쉽다.

나머지는 위 실행 결과를 보고 실제로 문제가 되는지 확인한 뒤 판단한다.

참고로 PR #10의 README에 RTX 5080에서 돌린 스모크 결과가 있다: word2vec/lstm **2 epoch에 accuracy 0.499**.
무작위 기준 0.05를 크게 넘으므로 **모델이 실제 데이터에서 학습은 된다.** 1번(padding)이 있더라도
학습을 완전히 막는 수준은 아니라는 뜻이다. 9-2 A/B는 '얼마나 손해를 보는지'를 재는 것이 된다.

### 다음에 할 것

- [ ] `SAMPLE_N = None`, `EPOCHS`를 늘려서 전체 데이터로 한 번 완주
- [ ] 9-2 A/B로 1번이 실제 문제인지 판별하고, 문제면 `model.py`에 반영
- [ ] LSTM vs GRU 비교 (`make_config(method, model_type="gru")`)
- [ ] 시드 3개로 반복해서 임베딩 방식 차이가 우연이 아닌지 확인